In [1]:
import json
import numpy as np
from pandas import read_csv
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, balanced_accuracy_score, cohen_kappa_score, matthews_corrcoef
import joblib
    

In [2]:
#Variables generales
ruta_model = "../../Model/"
ruta_metrics = "../../Metrics/"
ruta_df = "../../Datasets/"

semilla = 111

## FUNCIÓN BASE

In [3]:
def train_knn(X, y, output_model_path = 'best_model.pkl', output_metrics_path = 'metrics.json', kf = 2, param_grid = {'n_neighbors':[50]}):
    # Initialize KNeighborsClassifier
    knn = KNeighborsClassifier()
    
    # Define K-Fold Cross Validation
    kf = KFold(n_splits = kf
               , shuffle = True
               , random_state = 42
               )
    
    # Cross Validation
    grid_search = GridSearchCV(estimator = knn
                               , param_grid = param_grid
                               , cv = kf
                               , scoring = 'accuracy'
                               , n_jobs = -1
                               , verbose = 2
                               )
    
    # Ajustar el modelo
    grid_search.fit(X, y)
    
    # mejor modelo
    best_model = grid_search.best_estimator_
    
    # Guarde el mejor modelo en un archivo .pkl
    joblib.dump(best_model, output_model_path)
    
    # mejor modelo.
    y_pred = best_model.predict(X)
    
    # Calcular métricas de evaluación
    metrics = {
        'accuracy': accuracy_score(y, y_pred),
        'precision': precision_score(y, y_pred, average = 'weighted'),
        'recall': recall_score(y, y_pred, average = 'weighted'),
        'f1_score': f1_score(y, y_pred, average = 'weighted'),
        'roc_auc': roc_auc_score(y, y_pred, multi_class = 'ovr'),
        'confusion_matrix': confusion_matrix(y, y_pred).tolist(),
        'classification_report': classification_report(y, y_pred, output_dict = True),
        'balanced_accuracy': balanced_accuracy_score(y, y_pred),
        'cohen_kappa': cohen_kappa_score(y, y_pred),
        'matthews_corrcoef': matthews_corrcoef(y, y_pred)
    }
    
    # Guarde las métricas en un archivo JSON
    with open(output_metrics_path, 'w') as f:
        json.dump(metrics, f, indent = 4)

    return metrics, best_model

## Entrenamiento

### df interpolation

* SMOTE

In [4]:
# carga de caracteristicas
df_IM_smote = read_csv('{}Interpolation_Method_SMOTE_Train_Modificado.csv'.format(ruta_df))

In [5]:
param_grid = {
        'n_neighbors': [3, 5, 7, 9, 11],
        'weights': ['uniform', 'distance'],
        'algorithm': ['ball_tree', 'kd_tree', 'brute'],
        'p': [1, 2]
    }

metric_1, model_1 = train_knn(X = df_IM_smote.drop(columns='FLAG')
                              , y = df_IM_smote['FLAG']
                              , output_model_path = '{}KNN_IM_S.pkl'.format(ruta_model)
                              , output_metrics_path = '{}KNN_IM_S.json'.format(ruta_metrics)
                              , kf = 3
                              , param_grid = param_grid
                              )

Fitting 3 folds for each of 60 candidates, totalling 180 fits


In [6]:
metric_1

{'accuracy': 0.9949316384296254,
 'precision': 0.9949774591379023,
 'recall': 0.9949316384296254,
 'f1_score': 0.9949286130219432,
 'roc_auc': 0.9942980641007955,
 'confusion_matrix': [[27184, 0], [248, 21499]],
 'classification_report': {'0': {'precision': 0.9909594634004083,
   'recall': 1.0,
   'f1-score': 0.9954592060934524,
   'support': 27184},
  '1': {'precision': 1.0,
   'recall': 0.988596128201591,
   'f1-score': 0.9942653655829442,
   'support': 21747},
  'accuracy': 0.9949316384296254,
  'macro avg': {'precision': 0.9954797317002042,
   'recall': 0.9942980641007955,
   'f1-score': 0.9948622858381984,
   'support': 48931},
  'weighted avg': {'precision': 0.9949774591379023,
   'recall': 0.9949316384296254,
   'f1-score': 0.9949286130219432,
   'support': 48931}},
 'balanced_accuracy': 0.9942980641007955,
 'cohen_kappa': 0.989724839239456,
 'matthews_corrcoef': 0.9897770904210553}

* ADASYN

In [7]:
# carga de caracteristicas
df_IM_adasyn = read_csv('{}Interpolation_Method_ADASYN_Train_Modificado.csv'.format(ruta_df))

In [8]:
param_grid = {
        'n_neighbors': [3, 5, 7, 9, 11],
        'weights': ['uniform', 'distance'],
        'algorithm': ['ball_tree', 'kd_tree', 'brute'],
        'p': [1, 2]
    }

metric_2, model_2 = train_knn(X = df_IM_adasyn.drop(columns='FLAG')
                              , y = df_IM_adasyn['FLAG']
                              , output_model_path = '{}KNN_IM_A.pkl'.format(ruta_model)
                              , output_metrics_path = '{}KNN_IM_A.json'.format(ruta_metrics)
                              , kf = 3
                              , param_grid = param_grid
                              )

Fitting 3 folds for each of 60 candidates, totalling 180 fits


In [9]:
metric_2

{'accuracy': 0.9944536542436708,
 'precision': 0.9945084005448896,
 'recall': 0.9944536542436708,
 'f1_score': 0.9944499649341814,
 'roc_auc': 0.9937491350279097,
 'confusion_matrix': [[27184, 0], [271, 21406]],
 'classification_report': {'0': {'precision': 0.9901293024949918,
   'recall': 1.0,
   'f1-score': 0.9950401727703655,
   'support': 27184},
  '1': {'precision': 1.0,
   'recall': 0.9874982700558196,
   'f1-score': 0.9937098159366804,
   'support': 21677},
  'accuracy': 0.9944536542436708,
  'macro avg': {'precision': 0.9950646512474959,
   'recall': 0.9937491350279097,
   'f1-score': 0.994374994353523,
   'support': 48861},
  'weighted avg': {'precision': 0.9945084005448896,
   'recall': 0.9944536542436708,
   'f1-score': 0.9944499649341814,
   'support': 48861}},
 'balanced_accuracy': 0.9937491350279097,
 'cohen_kappa': 0.9887503396764933,
 'matthews_corrcoef': 0.9888129111947211}

### df linear regression

* SMOTE

In [10]:
# carga de caracteristicas
df_LR_smote = read_csv('{}Linear_Regression_SMOTE_Train_Modificado.csv'.format(ruta_df))

In [11]:
param_grid = {
        'n_neighbors': [3, 5, 7, 9, 11],
        'weights': ['uniform', 'distance'],
        'algorithm': ['ball_tree', 'kd_tree', 'brute'],
        'p': [1, 2]
    }

metric_3, model_3 = train_knn(X = df_LR_smote.drop(columns='FLAG')
                              , y = df_LR_smote['FLAG']
                              , output_model_path = '{}KNN_LR_S.pkl'.format(ruta_model)
                              , output_metrics_path = '{}KNN_LR_S.json'.format(ruta_metrics)
                              , kf = 3
                              , param_grid = param_grid
                              )

Fitting 3 folds for each of 60 candidates, totalling 180 fits


In [12]:
metric_3

{'accuracy': 0.9982015491201897,
 'precision': 0.998206804904088,
 'recall': 0.9982015491201897,
 'f1_score': 0.9982011962686702,
 'roc_auc': 0.9979859294319801,
 'confusion_matrix': [[27182, 2], [86, 21661]],
 'classification_report': {'0': {'precision': 0.9968461199941323,
   'recall': 0.9999264273101824,
   'f1-score': 0.9983838977448029,
   'support': 27184},
  '1': {'precision': 0.9999076766837465,
   'recall': 0.9960454315537776,
   'f1-score': 0.9979728173231975,
   'support': 21747},
  'accuracy': 0.9982015491201897,
  'macro avg': {'precision': 0.9983768983389394,
   'recall': 0.99798592943198,
   'f1-score': 0.9981783575340002,
   'support': 48931},
  'weighted avg': {'precision': 0.998206804904088,
   'recall': 0.9982015491201897,
   'f1-score': 0.9982011962686702,
   'support': 48931}},
 'balanced_accuracy': 0.99798592943198,
 'cohen_kappa': 0.996356725943439,
 'matthews_corrcoef': 0.9963627510635755}

* ADASYN

In [13]:
# carga de caracteristicas
df_LR_adasyn = read_csv('{}Linear_Regression_ADASYN_Train_Modificado.csv'.format(ruta_df))

In [14]:
param_grid = {
        'n_neighbors': [3, 5, 7, 9, 11],
        'weights': ['uniform', 'distance'],
        'algorithm': ['ball_tree', 'kd_tree', 'brute'],
        'p': [1, 2]
    }

metric_4, model_4 = train_knn(X = df_LR_adasyn.drop(columns='FLAG')
                              , y = df_LR_adasyn['FLAG']
                              , output_model_path = '{}KNN_LR_A.pkl'.format(ruta_model)
                              , output_metrics_path = '{}KNN_LR_A.json'.format(ruta_metrics)
                              , kf = 3
                              , param_grid = param_grid
                              )

Fitting 3 folds for each of 60 candidates, totalling 180 fits


In [15]:
metric_4

{'accuracy': 0.9992605525315805,
 'precision': 0.9992613124256339,
 'recall': 0.9992605525315805,
 'f1_score': 0.9992604946807915,
 'roc_auc': 0.9991725527555982,
 'confusion_matrix': [[27182, 2], [34, 21467]],
 'classification_report': {'0': {'precision': 0.998750734861846,
   'recall': 0.9999264273101824,
   'f1-score': 0.9993382352941176,
   'support': 27184},
  '1': {'precision': 0.9999068424239601,
   'recall': 0.9984186782010139,
   'f1-score': 0.9991622061903653,
   'support': 21501},
  'accuracy': 0.9992605525315805,
  'macro avg': {'precision': 0.999328788642903,
   'recall': 0.9991725527555981,
   'f1-score': 0.9992502207422415,
   'support': 48685},
  'weighted avg': {'precision': 0.9992613124256339,
   'recall': 0.9992605525315805,
   'f1-score': 0.9992604946807915,
   'support': 48685}},
 'balanced_accuracy': 0.9991725527555981,
 'cohen_kappa': 0.9985004421413826,
 'matthews_corrcoef': 0.9985013291753567}